# SpectralNet
Euclidean and Siamese affinities, following Section 3.3 and Appendix C of my thesis. These implementations were reconstructed in the earlier GitHub version; the original experiment notebook was not retained.

The network minimizes $\sum_{i,j} W_{ij}\|z_i-z_j\|^2$. An independent batch sets the orthogonalization layer before each gradient step. The final transform is fixed for inference, so embeddings from different batches share the same coordinates.

In [ ]:
%run ./functions.ipynb
from sklearn.neighbors import NearestNeighbors

seed = 42
dataset_name = 'mnist'       # also fashion_mnist, reuters10k, digits
n_samples = 2000             # None uses all available samples
n_clusters = 10              # use 4 for Reuters
hidden = [128, 64]
epochs = 10
siamese_epochs = 10
batch_size = 256
learning_rate = 1e-3
neighbors = 10
sigma = None                 # None: local kth-neighbor scale; float: fixed Gaussian scale
orthogonalization = 'cholesky'  # or 'qr'

X, y = load_data(dataset_name, n_samples=n_samples, seed=seed)
print(X.shape)

## Siamese network
Positive pairs are nearest neighbors in the input space. Negative pairs are distant points outside the positive-neighbor set. Class labels are used only in the result table.

In [ ]:
def make_network(dimensions):
    layers = []
    for i in range(len(dimensions) - 1):
        layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
        if i < len(dimensions) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)


class SiameseNet(nn.Module):
    def __init__(self, input_dim, hidden=(128, 64), output_dim=16):
        super().__init__()
        self.network = make_network([input_dim, *hidden, output_dim])

    def encode(self, x):
        return self.network(x)

    def forward(self, x):
        return self.encode(x)


def train_siamese(X, hidden=(128, 64), output_dim=16, epochs=10,
                  batch_size=256, lr=1e-3, neighbors=10, margin=1., seed=42):
    seed_everything(seed)
    X = torch.as_tensor(X, dtype=torch.float32).cpu()
    data = X.numpy()
    n = len(X)
    if n < 3 or neighbors < 1:
        raise ValueError('Need at least 3 samples and a positive neighbor count.')
    k = min(neighbors, n - 2)  # leave at least one possible negative
    nearest = NearestNeighbors(n_neighbors=k + 1).fit(data)
    indices = nearest.kneighbors(data, return_distance=False)
    positives = [row[row != i][:k] for i, row in enumerate(indices)]
    rng = np.random.default_rng(seed)
    model = SiameseNet(X.shape[1], hidden, output_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        total = 0.
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            anchors = order[start:start + batch_size]
            positive = np.array([rng.choice(positives[i]) for i in anchors])
            negative = []
            for i in anchors:
                forbidden = set(positives[i]) | {int(i)}
                candidates = np.array([j for j in rng.integers(n, size=64)
                                       if j not in forbidden], dtype=np.int64)
                if len(candidates) == 0:
                    candidates = np.array([j for j in range(n) if j not in forbidden])
                distances = ((data[candidates] - data[i]) ** 2).sum(axis=1)
                negative.append(candidates[distances.argmax()])
            a, p, q = [X[index].to(device) for index in (anchors, positive, negative)]
            za, zp, zq = model(a), model(p), model(q)
            positive_loss = (za - zp).square().sum(1)
            negative_loss = (margin - (za - zq).norm(dim=1)).clamp_min(0).square()
            loss = (positive_loss + negative_loss).mean()
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.)
            optimizer.step()
            total += loss.item() * len(a)
        history.append(total / n)
    model.eval()
    return model, history

## Affinity and orthogonalization
The graph is built within each minibatch. Directed k-nearest-neighbor weights are averaged with their transpose, and self-edges are removed. A fixed `sigma` gives the Gaussian kernel in Eq. 57; `sigma=None` uses the local-scale variant from the earlier repository.

Cholesky is the thesis choice. The optional QR calculation provides the same whitening constraint when the outputs have full column rank. Both normalize to $Z^T Z / b \approx I$; a small ridge term prevents singular transforms. Neither constructs a full MNIST affinity matrix.

In [ ]:
def affinity_matrix(x, neighbors=10, sigma=None):
    if len(x) < 2 or neighbors < 1:
        raise ValueError('An affinity batch needs at least 2 points and 1 neighbor.')
    d2 = torch.cdist(x, x).square()
    search = d2.clone()
    search.fill_diagonal_(float('inf'))
    values, indices = search.topk(min(neighbors, len(x) - 1), largest=False, dim=1)
    if sigma is None:
        scale = values[:, -1].sqrt().clamp_min(1e-6)
        denominator = 2 * scale[:, None] * scale[None, :]
    else:
        if sigma <= 0:
            raise ValueError('sigma must be positive.')
        denominator = 2 * sigma ** 2
    weights = torch.exp(-d2 / denominator)
    mask = torch.zeros_like(weights).scatter_(1, indices, 1.)
    weights = weights * mask
    weights = (weights + weights.T) / 2
    weights.fill_diagonal_(0.)
    return weights


class SpectralNet(nn.Module):
    def __init__(self, input_dim, n_clusters, hidden=(128, 64)):
        super().__init__()
        self.network = make_network([input_dim, *hidden, n_clusters])
        self.register_buffer('orthogonalizer', torch.eye(n_clusters))

    @torch.no_grad()
    def orthogonalize(self, x, method='cholesky'):
        output = self.network(x)
        if len(x) < output.shape[1]:
            raise ValueError('Orthogonalization batch must be at least n_clusters.')
        # Float64 makes the small k x k factorization less sensitive to roundoff.
        z = output.double()
        eye = torch.eye(z.shape[1], device=z.device, dtype=z.dtype)
        gram = z.T @ z / len(z)
        ridge = max(float(gram.diag().mean()) * 1e-7, 1e-10)
        if method == 'cholesky':
            for attempt in range(8):
                lower, info = torch.linalg.cholesky_ex(gram + ridge * eye)
                if int(info.max()) == 0:
                    break
                ridge *= 10
            if int(info.max()) != 0:
                raise FloatingPointError('Could not orthogonalize spectral outputs.')
            transform = torch.linalg.solve_triangular(lower, eye, upper=False).T
        elif method == 'qr':
            augmented = torch.cat([z / np.sqrt(len(z)), np.sqrt(ridge) * eye])
            _, upper = torch.linalg.qr(augmented, mode='reduced')
            transform = torch.linalg.solve_triangular(upper, eye, upper=True)
        else:
            raise ValueError('Choose cholesky or qr.')
        self.orthogonalizer.copy_(transform.to(output.dtype))

    def encode(self, x):
        return self.network(x) @ self.orthogonalizer

    def forward(self, x):
        return self.encode(x)

    @torch.no_grad()
    def predict(self, X, batch_size=256):
        if not hasattr(self, 'cluster_centers'):
            raise ValueError('Fit k-means on the spectral embeddings first.')
        self.eval()
        predictions = []
        for batch in torch.as_tensor(X, dtype=torch.float32).split(batch_size):
            z = self.encode(batch.to(self.orthogonalizer.device))
            predictions.append(torch.cdist(z, self.cluster_centers).argmin(1).cpu())
        return torch.cat(predictions).numpy()

In [ ]:
def train_spectralnet(X, n_clusters=10, hidden=(128, 64), epochs=10,
                      batch_size=256, lr=1e-3, neighbors=10, sigma=None,
                      siamese=False, siamese_epochs=10,
                      orthogonalization='cholesky', seed=42):
    seed_everything(seed)
    X = torch.as_tensor(X, dtype=torch.float32).cpu()
    if len(X) <= n_clusters or batch_size < 2:
        raise ValueError('Need more samples than clusters and batch_size >= 2.')
    metric_model, metric_history = None, []
    if siamese:
        metric_model, metric_history = train_siamese(
            X, hidden, epochs=siamese_epochs, batch_size=batch_size,
            lr=lr, neighbors=neighbors, seed=seed)
    model = SpectralNet(X.shape[1], n_clusters, hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        losses = []
        # array_split avoids a last batch containing just one sample.
        groups = max(1, len(X) // batch_size)
        for indices in np.array_split(rng.permutation(len(X)), groups):
            orth_indices = rng.choice(len(X), min(len(X), max(batch_size, n_clusters + 1)),
                                      replace=False)
            model.orthogonalize(X[orth_indices].to(device), orthogonalization)
            x = X[indices].to(device)
            with torch.no_grad():
                metric_x = metric_model(x) if metric_model is not None else x
                weights = affinity_matrix(metric_x, neighbors, sigma)
            z = model(x)
            loss = (weights * torch.cdist(z, z).square()).sum() / (2 * len(x) ** 2)
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite SpectralNet loss.')
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.)
            optimizer.step()
            losses.append(loss.item())
        history.append(float(np.mean(losses)))
    # Fix one transform for every subsequent inference batch.
    calibration = rng.choice(len(X), min(4096, len(X)), replace=False)
    model.orthogonalize(X[calibration].to(device), orthogonalization)
    model.eval()
    with torch.no_grad():
        Z = torch.cat([model.encode(batch.to(device)).cpu()
                       for batch in X.split(batch_size)]).numpy()
    km = KMeans(n_clusters, n_init=10, random_state=seed).fit(Z)
    model.register_buffer('cluster_centers', torch.tensor(km.cluster_centers_, device=device))
    return {'model': model, 'Z': Z, 'pred': km.labels_, 'history': history,
            'siamese': metric_model, 'siamese_history': metric_history}

## Compare the two affinities
The small defaults are for trying the code. Increase the data and training budget before comparing scores with the thesis. The same functions accept the Reuters features prepared in `reuters_fashion.ipynb`.

In [ ]:
results = {}
rows = []
for use_siamese in [False, True]:
    name = 'Siamese' if use_siamese else 'Euclidean'
    result = train_spectralnet(
        X, n_clusters, hidden, epochs, batch_size, learning_rate,
        neighbors, sigma, use_siamese, siamese_epochs,
        orthogonalization, seed)
    results[name] = result
    rows.append({'affinity': name, **score_dict(y, result['pred'])})
    plt.plot(result['history'], label=name)
plt.xlabel('Epoch')
plt.ylabel('Spectral loss')
plt.legend()
plt.show()
pd.DataFrame(rows)

In [ ]:
# The inference transform stays fixed, including for new batches.
predictions = results['Euclidean']['model'].predict(X, batch_size=batch_size)
print('Same predictions with batched inference:',
      np.array_equal(predictions, results['Euclidean']['pred']))